# Deep Nested UNets (Liu & Qian, TIFS 2021) - paper-aligned enhancement network
Changes vs the previous notebook: `DEPTH=7` (paper enhancement net), `SSIM_WIN=60` (paper 60x60 neighbourhood), fixed lr 1e-3 (no scheduler),
train until validation convergence (early stopping), best checkpoint by val SSIM, separate output folder, same 338-image evaluation protocol.
Not exact: same padding instead of valid convs, 256x256 crops, sigmoid output, much smaller training set (1764 pairs vs 30,000).

In [ ]:
# 1. Download + extract dataset (re-extracts if previous extraction was incomplete)
!pip -q install gdown
import os, zipfile, shutil, gdown
FILE_ID  = "1njl-hWUG7OEQVilJ4f8w77YuZe7EXrtM"
ZIP_PATH = "/kaggle/working/dataset.zip"
EXTRACT  = "/kaggle/working/dataset"
EXPECTED = 4984

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(f"{EXTRACT}/noise_project_v2") != EXPECTED:
    if not (os.path.exists(ZIP_PATH) and zipfile.is_zipfile(ZIP_PATH)):
        gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", ZIP_PATH, quiet=False)
    shutil.rmtree(EXTRACT, ignore_errors=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT)
n = n_files(f"{EXTRACT}/noise_project_v2")
print("files extracted:", n, "(expected", EXPECTED, ")")
assert n == EXPECTED, "extraction incomplete - check disk space with !df -h /kaggle/working"

In [ ]:
# 2. Imports + config (only edit this cell)
import os, json, time, random, shutil, subprocess
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
try:
    from skimage.metrics import structural_similarity as sk_ssim
except ImportError:
    subprocess.run(["pip", "install", "-q", "scikit-image"])
    from skimage.metrics import structural_similarity as sk_ssim

ROOT = Path("/kaggle/working/dataset/noise_project_v2")
CLEAN_DIR, NOISY_DIR = ROOT / "clean_500", ROOT / "noisy_500_v2"
OUT = Path("/kaggle/working/dnunet_paper"); OUT.mkdir(parents=True, exist_ok=True)   # new folder: baseline run stays untouched

SEED = 42
# ---- paper settings ----
DEPTH, CH = 7, 32              # enhancement net: 7 rows, 32 filters per layer
LR = 1e-3                      # Adam, fixed lr (no scheduler)
SSIM_WIN = 60                  # paper: 60x60 local neighbourhood
LAMBDA = 0.5                   # paper does not give lambda
# ---- training protocol (same rule for every model) ----
MAX_EPOCHS, PATIENCE, MIN_DELTA = 100, 10, 1e-4     # stop if val SSIM does not improve for 10 epochs
CROP, CROPS_PER_IMAGE, BATCH = 256, 2, 8
NUM_WORKERS = 2

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda")
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 3. Pair clean/noisy, official split, sanity checks
rows = []
for p in sorted(CLEAN_DIR.rglob("*.png")):
    rel = p.relative_to(CLEAN_DIR); n = NOISY_DIR / rel
    if n.exists():
        rows.append(dict(rel=str(rel), stem=p.stem, ftype=rel.parts[0],
                         participant=int(p.stem.split("_")[0]), clean=str(p), noisy=str(n)))
pairs = pd.DataFrame(rows)
print("pairs found:", len(pairs))

sp = pd.read_csv(ROOT / "b6000_split.csv")
sp.columns = [c.strip().lower() for c in sp.columns]
sp["split"] = (sp["split"].astype(str).str.strip().str.lower()
               .replace({"validation": "val", "valid": "val", "training": "train"}))

if "key" in sp.columns and sp["key"].astype(str).isin(pairs.stem).mean() > 0.9:
    pairs["split"] = pairs.stem.map(dict(zip(sp["key"].astype(str), sp["split"])))
    print("split mapped by 'key'")
else:
    sp["participant"] = pd.to_numeric(sp["participant"], errors="coerce")
    sp = sp.dropna(subset=["participant"]); sp["participant"] = sp["participant"].astype(int)
    assert sp.groupby("participant")["split"].nunique().max() == 1, "a participant is in >1 split"
    pairs["split"] = pairs.participant.map(sp.drop_duplicates("participant").set_index("participant")["split"])
    print("split mapped by 'participant'")

pairs = pairs.dropna(subset=["split"])
assert pairs.groupby("participant")["split"].nunique().max() == 1
train_df, val_df, test_df = [pairs[pairs.split == s].reset_index(drop=True) for s in ["train", "val", "test"]]
print("train/val/test:", len(train_df), len(val_df), len(test_df), "(expected 1764/388/338)")
assert (len(train_df), len(val_df), len(test_df)) == (1764, 388, 338), "split sizes differ - check cell 1 extraction"

bad = sum(cv2.imread(r.noisy, 0).shape != cv2.imread(r.clean, 0).shape
          for r in pairs.sample(min(100, len(pairs)), random_state=0).itertuples())
print("shape mismatches in 100-pair sample:", bad)

In [ ]:
# 4. Datasets + loaders
class TrainDS(Dataset):
    def __init__(self, df, reps): self.df, self.reps = df.reset_index(drop=True), reps
    def __len__(self): return len(self.df) * self.reps
    def __getitem__(self, i):
        r = self.df.iloc[i % len(self.df)]
        n = cv2.imread(r.noisy, cv2.IMREAD_GRAYSCALE); c = cv2.imread(r.clean, cv2.IMREAD_GRAYSCALE)
        H, W = n.shape
        ph, pw = max(0, CROP - H), max(0, CROP - W)
        if ph or pw:
            n = cv2.copyMakeBorder(n, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
            c = cv2.copyMakeBorder(c, 0, ph, 0, pw, cv2.BORDER_REFLECT_101)
            H, W = n.shape
        y, x = random.randint(0, H - CROP), random.randint(0, W - CROP)
        n, c = n[y:y+CROP, x:x+CROP], c[y:y+CROP, x:x+CROP]
        if random.random() < 0.5: n, c = n[:, ::-1], c[:, ::-1]
        f = lambda a: torch.from_numpy(np.ascontiguousarray(a)).float().div(255).unsqueeze(0)
        return f(n), f(c)

class EvalDS(Dataset):
    def __init__(self, df): self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        n = cv2.imread(r.noisy, cv2.IMREAD_GRAYSCALE); c = cv2.imread(r.clean, cv2.IMREAD_GRAYSCALE)
        f = lambda a: torch.from_numpy(a).float().div(255).unsqueeze(0)
        return f(n), f(c), i

train_loader = DataLoader(TrainDS(train_df, CROPS_PER_IMAGE), batch_size=BATCH, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=True)
val_loader  = DataLoader(EvalDS(val_df),  batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(EvalDS(test_df), batch_size=1, shuffle=False, num_workers=NUM_WORKERS)
print(len(train_loader), "train iters/epoch")

In [ ]:
# 5. Model: nested U-Nets + dense skips (5x5 convs, 32 filters, learned 2x2 stride-2 down/up, ReLU after every layer)
class Conv5(nn.Module):                       # 5x5 conv + ReLU
    def __init__(self, i, o): super().__init__(); self.c = nn.Conv2d(i, o, 5, padding=2)
    def forward(self, x): return F.relu(self.c(x), inplace=True)

class Down(nn.Module):                        # learned 2x2, stride 2 (replaces max-pool)
    def __init__(self, c): super().__init__(); self.c = nn.Conv2d(c, c, 2, stride=2)
    def forward(self, x): return F.relu(self.c(x), inplace=True)

class Up(nn.Module):                          # learned 2x2, stride 2 (replaces up-sampling)
    def __init__(self, c): super().__init__(); self.c = nn.ConvTranspose2d(c, c, 2, stride=2)
    def forward(self, x): return F.relu(self.c(x), inplace=True)

class DeepNestedUNet(nn.Module):
    """Node X[i,j] takes ALL same-row nodes X[i,0..j-1] plus the up-sampled X[i+1,j-1]; output read from X[0,depth-1]."""
    def __init__(self, in_ch=1, ch=32, depth=7):
        super().__init__()
        self.depth = depth
        self.stem = Conv5(in_ch, ch)
        self.down = nn.ModuleList([Down(ch) for _ in range(depth - 1)])
        self.enc  = nn.ModuleList([Conv5(ch, ch) for _ in range(depth - 1)])
        self.up, self.node = nn.ModuleDict(), nn.ModuleDict()
        for j in range(1, depth):
            for i in range(depth - j):
                self.up[f"{i}_{j}"]   = Up(ch)
                self.node[f"{i}_{j}"] = Conv5(ch * (j + 1), ch)
        self.out = nn.Conv2d(ch, 1, 5, padding=2)

    def forward(self, inp):
        x = {(0, 0): self.stem(inp)}
        for i in range(1, self.depth):
            x[(i, 0)] = self.enc[i-1](self.down[i-1](x[(i-1, 0)]))
        for j in range(1, self.depth):
            for i in range(self.depth - j):
                feats = [x[(i, k)] for k in range(j)] + [self.up[f"{i}_{j}"](x[(i+1, j-1)])]
                x[(i, j)] = self.node[f"{i}_{j}"](torch.cat(feats, 1))
        return torch.sigmoid(self.out(x[(0, self.depth - 1)]))

m = DeepNestedUNet(1, CH, DEPTH).to(device)
print("params: %.2f M" % (sum(p.numel() for p in m.parameters()) / 1e6))
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    print(m(torch.zeros(1, 1, 256, 256, device=device)).shape)
del m; torch.cuda.empty_cache()

In [ ]:
# 6. Loss (lambda*(1-SSIM_60x60) + (1-lambda)*MSE), metrics, full-image inference
def ssim_torch(x, y, win=SSIM_WIN):           # local SSIM, c1=(0.01)^2, c2=(0.03)^2, l=1
    c1, c2 = 0.01**2, 0.03**2
    mx, my = F.avg_pool2d(x, win, 1), F.avg_pool2d(y, win, 1)
    sxx = F.avg_pool2d(x*x, win, 1) - mx*mx
    syy = F.avg_pool2d(y*y, win, 1) - my*my
    sxy = F.avg_pool2d(x*y, win, 1) - mx*my
    return (((2*mx*my + c1) * (2*sxy + c2)) / ((mx*mx + my*my + c1) * (sxx + syy + c2))).mean()

def enhance_loss(p, t):
    p, t = p.float(), t.float()
    return LAMBDA * (1 - ssim_torch(p, t)) + (1 - LAMBDA) * F.mse_loss(p, t)

def to_u8(t): return (t.squeeze().detach().cpu().numpy() * 255).round().clip(0, 255).astype(np.uint8)

def metrics_u8(pred, gt):                     # same function for every model: uint8, 0-255 scale, full image
    d = pred.astype(np.float64) - gt.astype(np.float64)
    rmse = float(np.sqrt((d**2).mean()))
    psnr = 20 * np.log10(255 / rmse) if rmse > 0 else 100.0
    ssim = float(sk_ssim(gt, pred, data_range=255))
    return rmse, psnr, ssim

@torch.no_grad()
def predict_full(model, x):
    H, W = x.shape[-2:]; m = 2 ** (DEPTH - 1)
    xp = F.pad(x, (0, (-W) % m, 0, (-H) % m), mode="reflect")
    with torch.autocast("cuda", dtype=torch.float16):
        y = model(xp)
    return y[..., :H, :W].float().clamp(0, 1)

In [ ]:
# 7. Train until validation convergence (early stopping on val SSIM, patience 10, max 100 epochs), auto-resume
model = DeepNestedUNet(1, CH, DEPTH).to(device)
opt = torch.optim.Adam(model.parameters(), lr=LR)          # fixed lr, no scheduler (paper)
scaler = torch.amp.GradScaler("cuda")
LAST, BEST = OUT / "last.pt", OUT / "best.pt"
hist, start, best_ssim, bad = [], 0, -1.0, 0

if LAST.exists():
    ck = torch.load(LAST, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    hist, start, best_ssim, bad = ck["hist"], ck["epoch"] + 1, ck["best"], ck["bad"]
    print("resumed from epoch", start, "| best val SSIM", round(best_ssim, 4), "| no-improve", bad)

for ep in range(start, MAX_EPOCHS):
    t0 = time.time(); model.train(); tl, n = 0.0, 0
    for x, y in tqdm(train_loader, leave=False, desc=f"epoch {ep+1}"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16):
            p = model(x)
        loss = enhance_loss(p, y)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        tl += loss.item() * x.size(0); n += x.size(0)

    model.eval(); vl, R, P, S = 0.0, [], [], []
    for x, y, _ in val_loader:
        x, y = x.to(device), y.to(device)
        p = predict_full(model, x)
        vl += enhance_loss(p, y).item()
        r, ps, s = metrics_u8(to_u8(p), to_u8(y)); R.append(r); P.append(ps); S.append(s)
    rec = dict(epoch=ep+1, train_loss=tl/n, val_loss=vl/len(val_loader), val_rmse=np.mean(R),
               val_psnr=np.mean(P), val_ssim=np.mean(S), sec=time.time()-t0)
    hist.append(rec)

    if rec["val_ssim"] > best_ssim + MIN_DELTA:
        best_ssim, bad = rec["val_ssim"], 0
        torch.save(dict(model=model.state_dict(), epoch=ep + 1), BEST)
    else:
        bad += 1
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), hist=hist,
                    epoch=ep, best=best_ssim, bad=bad), LAST)
    json.dump(hist, open(OUT / "history.json", "w"))
    print(f"ep {ep+1:03d} | train {rec['train_loss']:.4f} | val {rec['val_loss']:.4f} | RMSE {rec['val_rmse']:.2f} "
          f"PSNR {rec['val_psnr']:.2f} SSIM {rec['val_ssim']:.4f} | best {best_ssim:.4f} | wait {bad}/{PATIENCE} | {rec['sec']:.0f}s")
    if bad >= PATIENCE:
        print("early stop: val SSIM did not improve for", PATIENCE, "epochs"); break

In [ ]:
# 8. Training curves
h = pd.DataFrame(json.load(open(OUT / "history.json")))
fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(h.epoch, h.val_psnr, c="C2"); ax[1].set_title("Val PSNR (dB)")
ax[2].plot(h.epoch, h.val_ssim, c="C3"); ax[2].set_title("Val SSIM")
ax[3].plot(h.epoch, h.val_rmse, c="C4"); ax[3].set_title("Val RMSE")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "training_curves.png", dpi=150); plt.show()
ck = torch.load(BEST, map_location="cpu", weights_only=False)
print(f"epochs run: {len(h)} | best epoch: {ck['epoch']} | total train time: {h.sec.sum()/60:.0f} min")

In [ ]:
# 9. Test on all 338 images: enhanced / noisy / clean saved the same way (uint8 PNG), PSNR/RMSE/SSIM
model = DeepNestedUNet(1, CH, DEPTH).to(device)
model.load_state_dict(torch.load(BEST, map_location=device, weights_only=False)["model"]); model.eval()

EV = OUT / "eval338"
shutil.rmtree(EV, ignore_errors=True)
D = {k: EV / k for k in ["noisy", "enhanced", "clean"]}
for d in D.values(): d.mkdir(parents=True)

recs = []
for x, y, i in tqdm(test_loader):
    r = test_df.iloc[int(i)]
    p = predict_full(model, x.to(device))
    pu, nu, cu = to_u8(p), to_u8(x), to_u8(y)
    nr, np_, ns = metrics_u8(nu, cu); er, ep_, es = metrics_u8(pu, cu)
    recs.append(dict(stem=r.stem, ftype=r.ftype, participant=r.participant,
                     noisy_rmse=nr, noisy_psnr=np_, noisy_ssim=ns,
                     enh_rmse=er, enh_psnr=ep_, enh_ssim=es))
    cv2.imwrite(str(D["enhanced"] / f"{r.stem}.png"), pu)
    cv2.imwrite(str(D["noisy"] / f"{r.stem}.png"), nu)
    cv2.imwrite(str(D["clean"] / f"{r.stem}.png"), cu)

res = pd.DataFrame(recs)
assert len(res) == 338
res.to_csv(OUT / "test_results.csv", index=False)
print(res[["noisy_rmse", "enh_rmse", "noisy_psnr", "enh_psnr", "noisy_ssim", "enh_ssim"]].mean().round(4))

In [ ]:
# 10. NFIQ2 CLI (skip install if already available)
import requests
NFIQ2_BIN = shutil.which("nfiq2") or shutil.which("NFIQ2")
if NFIQ2_BIN is None:
    rel = requests.get("https://api.github.com/repos/usnistgov/NFIQ2/releases/latest", timeout=30).json()
    cli = next(a for a in rel["assets"] if a["name"] == "nfiq2_2.3.0-1_ubuntu24_amd64.deb")
    subprocess.run(["wget", "-q", cli["browser_download_url"], "-O", "/tmp/nfiq2_cli.deb"], check=True)
    print(subprocess.run(["apt-get", "install", "-y", "/tmp/nfiq2_cli.deb"], capture_output=True, text=True).stdout[-400:])
    NFIQ2_BIN = shutil.which("nfiq2") or "/usr/local/bin/nfiq2"
print("NFIQ2 binary:", NFIQ2_BIN)

In [ ]:
# 11. NFIQ2 on the same 338 images (same command for noisy / enhanced / clean) + final table
def run_nfiq2(group):
    out_csv = EV / f"nfiq2_{group}.csv"
    p = subprocess.run([NFIQ2_BIN, "-F", "-i", str(D[group]), "-o", str(out_csv)], capture_output=True, text=True)
    assert out_csv.exists(), p.stderr[-800:]
    d = pd.read_csv(out_csv)
    fcol = next(c for c in d.columns if "file" in c.lower() or "name" in c.lower())
    qcol = next(c for c in d.columns if "quality" in c.lower() or "score" in c.lower())
    d["stem"] = d[fcol].astype(str).map(lambda s: Path(s).stem)
    d["s"] = pd.to_numeric(d[qcol], errors="coerce")
    d.loc[(d.s < 0) | (d.s > 100), "s"] = np.nan
    return d.set_index("stem")["s"]

for g in ["noisy", "enhanced", "clean"]:
    res[f"nfiq2_{g}"] = res.stem.map(run_nfiq2(g))
res.to_csv(OUT / "test_results_with_nfiq2.csv", index=False)
print("failed NFIQ2:", res[["nfiq2_noisy", "nfiq2_enhanced", "nfiq2_clean"]].isna().sum().to_dict())
print("sanity - noisy mean should be ~20.08, clean ~71.77:", round(res.nfiq2_noisy.mean(), 2), round(res.nfiq2_clean.mean(), 2))

ok = res.dropna(subset=["nfiq2_noisy", "nfiq2_enhanced", "nfiq2_clean"])
ms = lambda c: f"{ok[c].mean():.3f} +- {ok[c].std():.3f}"
summary = pd.DataFrame({
    "Noisy":    [ms("noisy_rmse"), ms("noisy_psnr"), ms("noisy_ssim"), ms("nfiq2_noisy")],
    "Enhanced": [ms("enh_rmse"),   ms("enh_psnr"),   ms("enh_ssim"),   ms("nfiq2_enhanced")],
    "Clean ref":["0", "-", "1", ms("nfiq2_clean")],
}, index=["RMSE (lower better)", "PSNR dB", "SSIM", "NFIQ2"])
print(f"n = {len(ok)} of {len(res)}"); display(summary)
summary.to_csv(OUT / "summary338.csv")
display(ok.groupby("ftype")[["noisy_psnr", "enh_psnr", "noisy_ssim", "enh_ssim", "nfiq2_noisy", "nfiq2_enhanced", "nfiq2_clean"]].mean().round(3))